In [1]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML
import csv

In [2]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [3]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [4]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [5]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [6]:
UNIVERSE_A = {}
UNIVERSE_O = {}
target_dates = {}

for ticker in INDEX_DATA:
    if ticker in a_share_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_A[ticker] = INDEX_DATA[ticker]

    elif ticker in other_market_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_O[ticker] = INDEX_DATA[ticker]

In [7]:
print(f"Assets under Analysis: {len(UNIVERSE_A)}  {len(UNIVERSE_O)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 73  35
Total Assets: 186


In [8]:
n = 0
for ticker in other_market_dict:
    if ticker in INDEX_DATA:
        n += 1

n

39

## Z Score Calculation

In [13]:
UNIVERSE = UNIVERSE_A
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 5
duration = years * 250

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [14]:
display_number = 20

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]

In [15]:
print()
print("A股")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


A股
2025-04-28
5年


,Ticker,指数名称,市盈率
9,399812.SZ,养老产业,-1.917391
37,h30035.CSI,300非银,-1.537020
58,399966.SZ,中证800证保,-1.492981
47,931068.CSI,消费龙头,-1.405564
50,930653.CSI,CS食品饮,-1.314724
45,931139.CSI,CS消费50,-1.285276
10,000815.CSI,细分食品,-1.266564
46,931079.CSI,5G通信,-1.165280
49,931008.CSI,汽车指数,-1.155797
7,707717L.MI,MSCI中国A股质量,-1.151551


In [25]:
UNIVERSE = UNIVERSE_O
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 5
duration = years * 250

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [26]:
display_number = len(other_market_dict)

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]

In [27]:
print()
print("海外")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


海外
2025-04-28
5年


,Ticker,指数名称,市盈率
7,931787CNY00.CSI,港股创新药,-1.337845
22,931024.CSI,港股非银,-1.334133
21,930604.CSI,中概互联,-1.251121
12,HSCGSI.HI,恒生消费,-0.784924
10,122489.MI,MSCI金龙,-0.773699
13,HSSCNE.HI,恒生新经济,-0.738412
26,TPX.GI,日本东证指数,-0.725810
5,N225.GI,日经225,-0.633774
17,CES100.CSI,港股通100,-0.552884
0,NDX.GI,纳斯达克指数,-0.466771
